# 1. Create UC volume (Catalog > Schema > Volumes tab)
# Path becomes: /Volumes/catalog/schema/bronze/

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS dev_catalog;
CREATE SCHEMA IF NOT EXISTS dev_catalog.pipeline_schema;  
CREATE VOLUME dev_catalog.pipeline_schema.bronze_table;   

In [0]:
%sql
-- drop volume if exists dev_catalog.pipeline_schema.bronze_table;
-- drop table dev_catalog.pipeline_schema.silver_table;
-- drop table dev_catalog.pipeline_schema.gold_table_daily_trends;
-- drop table dev_catalog.pipeline_schema.gold_table_status;
-- drop table dev_catalog.pipeline_schema.quarantine_table;
-- drop table dev_catalog.pipeline_schema.api_transformed;
-- drop table dev_catalog.pipeline_schema.dq_metrics;

In [0]:
%sql
-- select count(*) from dev_catalog.pipeline_schema.bronze_table;
select count(*) from dev_catalog.pipeline_schema.quarantine_table;
select count(*) from dev_catalog.pipeline_schema.api_transformed;
select count(*) from dev_catalog.pipeline_schema.dq_metrics;
select count(*) from dev_catalog.pipeline_schema.silver_table;
select count(*) from dev_catalog.pipeline_schema.gold_table_daily_trends;
select count(*) from dev_catalog.pipeline_schema.gold_table_status;



In [0]:
table_df = spark.read.format("delta") \
    .load("/Volumes/dev_catalog/pipeline_schema/bronze_table/api_data")
    
print(f"Total rows in Delta table: {table_df.count()}")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dev_catalog.pipeline_schema.pipeline_metadata (
    pipeline_name STRING,
    last_processed_timestamp TIMESTAMP
)
""")

In [0]:
spark.sql("""
INSERT INTO dev_catalog.pipeline_schema.pipeline_metadata
SELECT 'api_pipeline', TIMESTAMP('2000-01-01')
WHERE NOT EXISTS (
    SELECT 1 FROM dev_catalog.pipeline_schema.pipeline_metadata 
    WHERE pipeline_name = 'api_pipeline'
)
""")